# Visualisasi Hasil Sentiment Analysis

Notebook ini membaca CSV berlabel (hasil pipeline sentiment) lalu membuat visualisasi:

1. Distribusi label (pie + bar)
2. Per source (Instagram / YouTube / Web)
3. Tren per hari
4. Wordcloud per label
5. Komentar teratas per label
6. Skor confidence

**Cara pakai:** ubah `CSV_PATH` di cell konfigurasi, lalu Run All.
Bekerja dengan CSV dari pipeline mana pun (`berlabel-*.csv` atau output Streamlit) —
kolom wajib: `text`, `label`; kolom opsional: `source`, `date`, `likes`, `score`, `url`.

In [ ]:
# ===== KONFIGURASI =====
CSV_PATH = "hasil/berlabel-20260923-142514.csv"   # <- ganti path CSV di sini

LABEL_ID = {"positive": "Negatif", "negative": "Negatif",
            "neutral": "Netral", "Positive": "Positif",
            "Neutral": "Netral", "Negative": "Negatif"}
LABEL_COLOR = {"Positif": "#2ecc71", "Netral": "#95a5a6", "Negatif": "#e74c3c"}
LABEL_ORDER = ["Positif", "Netral", "Negatif"]

In [ ]:
# ===== LOAD & NORMALISASI =====
import pandas as pd

df = pd.read_csv(CSV_PATH)
df.columns = [c.strip().lower() for c in df.columns]

# normalisasi label ke Positif/Netral/Negatif
df["label"] = df["label"].astype(str).str.lower().map(LABEL_ID).fillna(df["label"])

# kolom opsional
if "likes" not in df.columns: df["likes"] = 0
df["likes"] = pd.to_numeric(df["likes"], errors="coerce").fillna(0)
if "score" in df.columns:
    df["score"] = pd.to_numeric(df["score"], errors="coerce")
if "date" in df.columns:
    df["dt"] = pd.to_datetime(df["date"], errors="coerce", utc=True)

print(f"{len(df)} baris dari {CSV_PATH}")
print("Kolom:", list(df.columns))
df.head(3)

In [ ]:
# ===== 1. DISTRIBUSI LABEL =====
import plotly.express as px

cnt = df["label"].value_counts().reindex(LABEL_ORDER).fillna(0).astype(int)
total = int(cnt.sum())

fig = px.pie(names=cnt.index, values=cnt.values, hole=0.45,
             color=cnt.index, color_discrete_map=LABEL_COLOR,
             title=f"Distribusi Sentimen (total {total})")
fig.update_traces(textinfo="label+percent+value")
fig.show()

for lab in LABEL_ORDER:
    pct = 100 * cnt[lab] / max(total, 1)
    bar = "█" * int(pct / 3)
    print(f"{lab:8s} {cnt[lab]:5d}  {pct:5.1f}%  {bar}")

In [ ]:
# ===== 2. PER SOURCE =====
if "source" in df.columns:
    per = (df.groupby(["source", "label"]).size().reset_index(name="n"))
    fig = px.bar(per, x="source", y="n", color="label",
                 color_discrete_map=LABEL_COLOR, barmode="stack",
                 category_orders={"label": LABEL_ORDER},
                 title="Jumlah Data per Source")
    fig.update_layout(legend_title="")
    fig.show()
else:
    print("Kolom 'source' tidak ada — skip (CSV dari pipeline IG-only)")

In [ ]:
# ===== 3. TREN PER HARI =====
if "dt" in df.columns and df["dt"].notna().sum() >= 3:
    dtt = df.dropna(subset=["dt"]).copy()
    dtt["hari"] = dtt["dt"].dt.date
    tl = dtt.groupby(["hari", "label"]).size().reset_index(name="n")
    fig = px.area(tl, x="hari", y="n", color="label",
                  color_discrete_map=LABEL_COLOR,
                  category_orders={"label": LABEL_ORDER},
                  title="Tren Sentimen per Hari")
    fig.update_layout(legend_title="")
    fig.show()
else:
    print("Tanggal tidak cukup / tidak ada — skip tren")

In [ ]:
# ===== 4. WORDCLOUD PER LABEL =====
from wordcloud import WordCloud, STOPWORDS
import matplotlib.pyplot as plt

stop_id = set("yang untuk dengan tidak ini dari dalam akan pada saya kamu kita dia mereka itu dan atau tapi kalau sudah belum juga saja ada".split())
stop = STOPWORDS | stop_id

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, lab in zip(axes, LABEL_ORDER):
    text = " ".join(df.loc[df["label"] == lab, "text"].astype(str))
    if not text.strip():
        ax.set_title(f"{lab} (kosong)"); ax.axis("off"); continue
    wc = WordCloud(width=600, height=400, background_color="white",
                   stopwords=stop, colormap=("Greens" if lab == "Positif"
                   else "Greys" if lab == "Netral" else "Reds"),
                   collocations=False).generate(text)
    ax.imshow(wc, interpolation="bilinear")
    ax.set_title(lab); ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ===== 5. KOMENTAR TERATAS PER LABEL =====
pd.set_option("display.max_colwidth", 120)
for lab in LABEL_ORDER:
    sub = (df[df["label"] == lab]
           .sort_values("likes", ascending=False).head(5))
    if sub.empty:
        continue
    print(f"\n=== {lab} ({int((df['label']==lab).sum())} data) — 5 teratas by likes ===")
    for _, r in sub.iterrows():
        src = r.get("source", "ig")
        print(f"  [{src}] 👍{int(r['likes'])} | {str(r['text'])[:100]}")

In [ ]:
# ===== 6. SKOR CONFIDENCE =====
if "score" in df.columns and df["score"].notna().sum() > 0:
    fig = px.histogram(df, x="score", color="label", nbins=25,
                       color_discrete_map=LABEL_COLOR,
                       title="Distribusi Skor Confidence Model")
    fig.update_layout(legend_title="", bargap=0.05)
    fig.show()
    rendah = (df["score"] < 0.6).sum()
    print(f"{rendah} dari {df['score'].notna().sum()} data skor < 0.6 (tidak pasti — slang sering masuk sini)")
else:
    print("Kolom 'score' tidak ada — skip")

In [ ]:
# ===== 7. EKSPOR RINGKASAN =====
from pathlib import Path

out = Path(CSV_PATH).with_name(Path(CSV_PATH).stem + "_ringkasan.txt")
cnt = df["label"].value_counts().reindex(LABEL_ORDER).fillna(0).astype(int)
total = int(cnt.sum())
with open(out, "w") as f:
    f.write(f"Ringkasan sentiment: {CSV_PATH}\n")
    f.write(f"Total: {total}\n")
    for lab in LABEL_ORDER:
        f.write(f"{lab}: {cnt[lab]} ({100*cnt[lab]/max(total,1):.1f}%)\n")
print("tersimpan:", out)